# Step-by-step pipeline to transform Master Site data into PyTorch Geometric format.


In [5]:
from itertools import combinations
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.neighbors import BallTree
from sklearn.pipeline import Pipeline
from torch_geometric.data import Data
from torch_geometric.utils import *
from sklearn.ensemble import RandomForestClassifier
from math import radians
from typing import *
from pandas import *
from io import *
import os
import glob
import json
# import pandas as pd
import numpy as np
import re
import random
import torch
import networkx as nx
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import plotly.graph_objects as go


# Pandas option
set_option("display.max_rows", 100000)       # increase width of display
set_option("display.width", 200)       # increase width of display
set_option("display.max_columns", None)  # show all columns
set_option("display.expand_frame_repr", False)  # stop wrapping with '\'


dateColumnName = 'DATE'
statusColumnName = 'STATUS'
retColumName = 'RET_NONRET'
regionColumnName = 'REGION'
kabupatenColumnName = 'KABUPATEN'
kecamatanColumnName = 'KECAMATAN'
antennaModelColumnName = 'ANTENNA_MODEL'
siteIdColumnName = 'SITEID'
siteTypeColumnName = 'SITE_TYPE'
antennaHeightColumnName = 'ANTENNA_HEIGHT'
longitudeColumnName = 'LONGITUDE'
latitudeColumnName = 'LATITUDE'
azimuthColumnName = 'AZIMUTH'
azimuthSinColumnName = 'AZIMUTH_SIN'
azimuthCosColumnName = 'AZIMUTH_COS'
horizontalBeamColumnName = 'HORIZONTAL_BEAM_WIDTH'
mtColumnName = 'MT'
etColumnName = 'ET'
cellIdColumName = 'CELLID'
modCellIdColumName = 'MOD_CELL_ID'
cellColumnName = 'CELLNAME'
technologyColumnName = 'TECHNOLOGY'
lacTacColumnName = 'LAC/TAC'
enodebColumnName = 'ENODEB_ID/NODEB_ID/BTS_ID'
gnbidColumnName = 'GNBID-BTS_ID'
vendorColumnName = 'VENDOR'
bandColumnName = 'BAND'
nodeIndexColumnName = 'NODE_INDEX'

<h3>Master Site initial data</h3>


In [6]:
# Load raw CSV file
source_dir = "/data/geolocation-data/raw/master_site"
source_files = glob.glob(source_dir + "/**/*.csv", recursive=True)


listOfDF: Optional[List[DataFrame]] = []
df_temp: Optional[DataFrame] = None
df_initial: Optional[DataFrame] = None
dir_path: Optional[str] = None


# this function intended to replace empty string, 'nan', 'null' value with 'NaN'
def replaceNACharacter(value):
    modValue: Optional[str] = None

    # validate if value is string
    if (isinstance(value, str)):
        modValue = value.strip().strip('"')  # remove leading/trailing spaces and remove excess '#' character

        if modValue.lower() in {"", "nan", "null"}:
            return NA

        return value

    return value


# remove excess '"' character inside specific column data


for file in source_files:
    dir_path = os.path.basename(os.path.dirname(file))

    # dir_path value is using date pattern of 'yyyy-mm-dd'

    if (dir_path is not None):
        df_temp = read_csv(
            file,
            low_memory=False,
            na_values=["", " ", "NULL", "NaN", "nan"],  # empty string, space, NULL and NaN value will be normalized as NaN object
            keep_default_na=True    # always use default value ---> 'NaN'
        )

        df_temp.map(replaceNACharacter)

        # add new column
        df_temp[dateColumnName] = dir_path

        listOfDF.append(df_temp)


# concat all dataframe into one
df_initial = concat(listOfDF, ignore_index=True)

if (df_initial is not None and not df_initial.empty):
    print("Found files:", len(source_files))
    print(f"Rows : {len(df_initial)}")

Found files: 9
Rows : 3365348


<h3>Normalize Master Site columns name</h3>


In [ ]:
listOfNewColumn: Optional[str] = []
tempName: Optional[str] = None

if (df_initial is not None and not df_initial.empty):

    # replace existing name with clean name
    for column in df_initial.columns:
        # replace character ' / ' with '-'
        tempName = column.upper().replace(" / ", "-")

        # replace empty space character with '_'
        # tempName = tempName.replace(" ", "_")
        tempName = re.sub(r"\s+", "_", tempName)

        listOfNewColumn.append(tempName)

    # replace the column name inside DataFrame
    df_initial.columns = listOfNewColumn

    print(f"Rows : {len(df_initial)}")
    print(f"Cols : {len(df_initial.columns)} \n\n\n")
    print(df_initial.head(20))

<h3>Normalized Master Site data</h3>


In [ ]:
# setOfCellNameSuffix: Optional[set[str]] = set()

df_normalize: Optional[DataFrame] = None


# Sector-0 randomize from [10,20,30,290,300] degree
# Sector-1 40~50 degree
# Sector-2 130~150 degree
# Sector-3 260~280 degree
sectorDict: Optional[Dict[int, any]] = {
    0: random.choice([10, 20, 30, 290, 300]),
    1: random.randint(40, 50),
    2: random.randint(130, 150),
    3: random.randint(260, 280),
}


def normalizeRetValue(value: str) -> str:
    # valid value : RET, NO

    result: Optional[str] = None

    try:
        if (value is not None):
            if (isna(value)):
                # print('value = NULL')
                result = 'NO'
            else:
                value = str(value).strip().upper()

                if (value == ""):
                    # empty string return NO
                    result = 'NO'

                elif (value in ['RET', 'NO']):
                    # return the value as it is
                    result = value

                else:
                    # other value return NO
                    result = 'NO'
    except Exception as e:
        print(f"---------- normalizeRetValue =====> Error occurred : {e}")
    else:
        return result
    finally:
        result = None


def normalizeAzimuthValue(value: str) -> str:

    suffix: Optional[str] = None
    sectors: Optional[int] = None
    result: Optional[int] = None
    extractedDigit: Optional[int] = None
    hasNonDigit: Optional[bool] = False
    hasDigit: Optional[bool] = False

    try:
        if (value is not None):

            value = str(value).strip()

            if (value != "" and value != 'NaN'):
                # retrieve last 2 character
                suffix = value[-2:]     # possible value, e.g : E3, G6, T1, BG, MI, UA, 2_, etc

                # check whether the suffix has non digit character
                hasNonDigit = any(not i.isdigit() for i in suffix)

                # check whether the suffix has digit character
                hasDigit = any(i.isdigit() for i in suffix)

                if (hasNonDigit):     # suffix has non digit character
                    # setOfCellNameSuffix.add(suffix)

                    # suffix also has digit character
                    if (hasDigit):

                        # extract the digit only
                        extractedDigit = int(re.sub(r"\D", "", suffix))

                        if (extractedDigit is not None):
                            sector = (int(extractedDigit) % 3)

                    # suffix contain no digit at all
                    else:
                        sector = 0

                else:  # suffix only have digit character
                    sector = (int(suffix) % 3)   # convert the value to int and then do modulus 3

    except Exception as e:
        print(f"---------- normalizeAzimuthValue =====> Error occurred : {e}")
    else:
        # print(f'sector : {sector}')
        result = sectorDict.get(sector)
    finally:
        suffix = None
        sector = None

        return result


def normalizeCellIdValue(technology: str, lac_tac: str, enodebId: str, gnbId: str, cellId: str) -> str:
    uniqueCellId: Optional[str] = None

    try:
        if (technology is not None and lac_tac is not None and enodebId is not None and cellId is not None):
            technology = str(technology).strip().upper()

            if (not isna(technology) and technology != ""):

                if (isinstance(cellId, str) or isna(cellId)):
                    cellId = str(cellId).strip().lower()
                else:
                    cellId = str(int(cellId))

                if (isinstance(lac_tac, str) or isna(lac_tac)):
                    lac_tac = str(lac_tac).strip().lower()
                else:
                    lac_tac = str(int(lac_tac))

                if (isinstance(enodebId, str) or isna(enodebId)):
                    enodebId = str(enodebId).strip().lower()
                else:
                    enodebId = str(int(enodebId))

                if (isinstance(gnbId, str) or isna(gnbId)):
                    gnbId = str(gnbId).strip().lower()
                else:
                    gnbId = str(int(gnbId))

                match(technology):
                    # 2G
                    case '2G':
                        if (not isna(lac_tac) and lac_tac != ''):
                            uniqueCellId = lac_tac + '_' + cellId
                        else:
                            uniqueCellId = cellId

                    # 4G
                    case '4G':
                        if (not isna(enodebId) and enodebId != ''):
                            uniqueCellId = enodebId + '_' + cellId
                        else:
                            uniqueCellId = cellId

                    # 5G
                    case '5G':
                        if (not isna(gnbId) and gnbId != ''):
                            uniqueCellId = gnbId + '_' + cellId
                        else:
                            uniqueCellId = cellId

                    case _:
                        uniqueCellId = cellId

    except Exception as e:
        print(f"---------- normalizeCellIdValue =====> Error occurred : {e}")
    finally:
        pass

    return uniqueCellId


# delete row if any column define below = NaN
df_normalize = df_initial.dropna(subset=[cellIdColumName, antennaHeightColumnName, latitudeColumnName, longitudeColumnName], how='any').copy()

# ======================================================= Normalize ret_nonret column data========================================================
df_normalize[retColumName] = df_normalize[retColumName].apply(normalizeRetValue)


# ========================================================== Normalized MT and ET value ==========================================================
# if antenna height below 20 m then
#   validate MT and ET, if MT has NaN value, then set MT = 5, if ET has NaN value, then set ET : 5

# if antenna height above 20 m then
#   validate MT and ET, if MT has NaN value, then set MT = 2, if ET has NaN value then set ET : 5

df_normalize.loc[(df_normalize[antennaHeightColumnName] < 20) & (df_normalize[mtColumnName].isna()), mtColumnName] = 5
df_normalize.loc[(df_normalize[antennaHeightColumnName] < 20) & (df_normalize[etColumnName].isna()), etColumnName] = 5
df_normalize.loc[(df_normalize[antennaHeightColumnName] >= 20) & (df_normalize[mtColumnName].isna()), mtColumnName] = 2
df_normalize.loc[(df_normalize[antennaHeightColumnName] >= 20) & (df_normalize[etColumnName].isna()), etColumnName] = 5


# =========================================================== Normalized Azimuth value ===========================================================
df_normalize[azimuthColumnName] = df_normalize.apply(
    lambda row: (
        normalizeAzimuthValue(
            row[cellColumnName]
        ) if isna(row[azimuthColumnName])
        else row[azimuthColumnName]
    ),
    axis=1
)


# ==================================================== Normalized azimuth to az-sin and az-cos ====================================================
df_normalize[azimuthSinColumnName] = np.sin(np.radians(df_normalize[azimuthColumnName].fillna(0)))
df_normalize[azimuthCosColumnName] = np.cos(np.radians(df_normalize[azimuthColumnName].fillna(0)))

# format them in decimals
df_normalize[[azimuthSinColumnName, azimuthCosColumnName]] = df_normalize[[azimuthSinColumnName, azimuthCosColumnName]].map(
    lambda x: (
        float(f"{x:.7f}")
    )
)

# ========================================================== Normalized cellId value =============================================================
df_normalize[modCellIdColumName] = df_normalize.apply(
    lambda row: (
        normalizeCellIdValue(
            row[technologyColumnName],
            row[lacTacColumnName],
            row[enodebColumnName],
            row[gnbidColumnName],
            row[cellIdColumName]
        )
    ), axis=1

)


# ================================================================================================================================================


if (df_normalize is not None and not df_normalize.empty):
    print(f"Rows : {len(df_normalize)}")
    print(f"Cols : {len(df_normalize.columns)} \n\n\n")
    print(df_normalize.head(20))

<h3>Filtering master site latest data by Date</h3>


In [ ]:
filter: Optional[Series] = None
df_allRegion: Optional[DataFrame] = None
dates: Optional[np.ndarray] = None

# retrieve latest date
dates = df_initial[dateColumnName].unique()
dates.sort()
latestDate = dates[-1]  # retrieve latest

# print(f'latest date : {latestDate}')

filter = (df_normalize[dateColumnName] == latestDate)    # the df will contain False / True

if (filter is not None and not filter.empty):
    df_allRegion = df_normalize[filter].copy()    # only copy data which contain True value

if (df_allRegion is not None and not df_allRegion.empty):
    print(f"Rows : {len(df_allRegion)}")
    print(f"Cols : {len(df_allRegion.columns)} \n\n\n")
    print(df_allRegion.head(20))

<h3>Prepare the Node and Edges index For Graph</h3>


In [ ]:
# This scenario will use Cell as the Node

df_index: Optional[DataFrame] = None
unique_cells: Optional[np.ndarray] = None
cellDict: Optional[Dict[str, int]] = {}


# make a dataset copy
# df_index = df_allRegion.copy()
df_index = df_region.copy()

# convert the column type to string to ensure index consistency
df_index[modCellIdColumName] = df_index[modCellIdColumName].astype(str)

# retrieve distinct value based on Cell columns
unique_cells = df_index[modCellIdColumName].unique()

if (unique_cells is not None and len(unique_cells) != 0):
    print("Unique nodes:", len(unique_cells))

    # map the cellId to number_index (string: int)
    for i, cellId in enumerate(unique_cells):
        cellDict[cellId] = i


# df_filter = df_index[df_index[['azimuth', 'mt', 'et']].isna().any(axis=1)]
# print(df_filter.head(50))


# add a new column with the value retrieve from Dict, using cellid as key
df_index[nodeIndexColumnName] = df_index[modCellIdColumName].map(cellDict)

if (df_index is not None and not df_index.empty):
    print(f"Rows : {len(df_index)}")
    print(f"Cols : {len(df_index.columns)} \n\n\n")
    # print(df_index[[modCellIdColumName, nodeIndexColumnName]].head(20))
    print(df_index.head(20))
    df_index.to_csv("/data/geolocation-data/processed/master_site/result.csv", index=False)

In [ ]:
# this is just dummy step
source_file = '/data/geolocation-data/processed/master_site/result.csv'

df_dummy = read_csv(
    source_file,
    low_memory=False,
    na_values=["", " ", "NULL", "NaN", "nan"],  # empty string, space, NULL and NaN value will be normalized as NaN object
    keep_default_na=True    # always use default value ---> 'NaN'
)


if (df_dummy is not None and not df_dummy.empty):

    distinct_count = df_dummy["SITEID"].nunique()
    print(f'distinct siteId : {distinct_count}')

    print(f"Rows : {len(df_dummy)}")
    print(f"Cols : {len(df_dummy.columns)} \n\n\n")
    print(df_dummy.head(20))

<h3>Prepare the Node Features For GCN</h3>


In [ ]:
# Features selection


df_feature: Optional[DataFrame] = None
lowCardCatCols: Optional[List] = None
lowCardNumCols: Optional[List] = None
highCardCatCols: Optional[List] = None
highCardNumCols: Optional[List] = None
numericalCols: Optional[List] = None
scaler: Optional[StandardScaler] = None
X_num: Optional[torch.Tensor] = None
node_features: Optional[np.ndarray] = None

cellid_to_idx: Optional[Dict[int, int]] = None
idx_to_cellid: Optional[Dict[int, int]] = None
lowCardCatTensors: Optional[Dict[str, torch.Tensor]] = None
highCardCatTensors: Optional[Dict[str, torch.Tensor]] = None

processedDataDir = '/data/geolocation-data/processed/master_site'
processedMasterSiteFile = 'result.csv'

# enabled this to refer from previous process DataFrame
# df_feature = df_index.copy()


# enabled this to read from exported file
df_feature = read_csv(
    os.path.join(processedDataDir, processedMasterSiteFile),
    low_memory=False,
    na_values=["", " ", "NULL", "NaN", "nan"],  # empty string, space, NULL and NaN value will be normalized as NaN object
    keep_default_na=True    # always use default value ---> 'NaN'
)


# ==============================
# 1. numerical features
# ==============================

# fit() → calculates the mean and standard deviation for each numeric column.
# transform() → applies the standardization using those values.

numericalCols = [antennaHeightColumnName, mtColumnName, etColumnName, azimuthSinColumnName, azimuthCosColumnName, longitudeColumnName, latitudeColumnName, horizontalBeamColumnName]

scaler = StandardScaler()
df_feature[numericalCols] = scaler.fit_transform(df_feature[numericalCols])

X_num = torch.tensor(df_feature[numericalCols].values, dtype=torch.float)


# ========================================
# 2. low-cardinality Categorical features
# ========================================
lowCardCatCols = [technologyColumnName, vendorColumnName, bandColumnName, siteTypeColumnName, regionColumnName, retColumName]

# convert the each category to unique numeric value
for col in lowCardCatCols:
    df_feature[f'{col}_code'] = df_feature[col].astype('category').cat.codes


lowCardCatTensors = {
    col: torch.tensor(df_feature[f'{col}_code'].values, dtype=torch.long) for col in lowCardCatCols
}


# ========================================
# 3. high-cardinality Categorical features
# ========================================
highCardCatCols = [kabupatenColumnName, kecamatanColumnName, antennaModelColumnName]

# convert the each category to unique numeric value
for col in highCardCatCols:
    df_feature[f'{col}_code'] = df_feature[col].astype('category').cat.codes


highCardCatTensors = {
    col: torch.tensor(df_feature[f'{col}_code'].values, dtype=torch.long) for col in highCardCatCols
}


# --------------------------------------
# 4. Node index (critical for GCN)
# --------------------------------------

# below dict needed for constructing edge_index

# this dict map modCellid : node_index
cellid_to_idx = dict(zip(df_feature[modCellIdColumName], df_feature[nodeIndexColumnName]))

# print("Node count:", len(cellid_to_idx))
# for k, v in cellid_to_idx.items():
#     print(f'{k} : {v}')

# this dict map node_index : cellid
idx_to_cellid = dict(zip(df_feature[nodeIndexColumnName], df_feature[modCellIdColumName]))
# for k, v in idx_to_cellid.items():
#     print(f'{k} : {v}')


# add the features to node_features
lowCardNumCols = [f'{col}_code' for col in lowCardCatCols]
highCardNumCols = [f'{col}_code' for col in highCardCatCols]


node_features = df_feature[numericalCols + lowCardNumCols + highCardNumCols].values
print("Node feature matrix shape:", node_features.shape)

<h1></h1>


<h3>Scenario A - Cell connection with within same site (IntraSite)</h3>


In [ ]:
edge_list: Optional[List[tuple[int, int]]] = []
node_indices: Optional[List[int]] = None


grouped_list = []


# Group by siteid and create edges among all cells within the same site
for siteId, group in df_feature.groupby(siteIdColumnName):
    # print(siteId)
    # print(f'group : \n {group}\n\n')

    # add the group to list
    grouped_list.append(group)

    node_indices = group[nodeIndexColumnName].tolist()
    # print(node_indices)

    if len(node_indices) > 1:

        # create a pair combination between each item in the list
        edge_list.extend(list(combinations(node_indices, 2)))


print(f'length of edge_list : {len(edge_list)}')
# for i in edge_list:
#     print(i)

# Convert edge list to tensor format for PyTorch Geometric
edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
print(f"shape of edge_index after transpose : {edge_index.shape}")

# Make edges bidirectional (GCN expects undirected graph)
edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
print(f"shape of edge_index after flip and concat : {edge_index.shape}")

# print(f"Shape of edge_index : {edge_index.shape}")
# print(f"Total edges (including both directions) : {edge_index.shape[1]}\n")


# Build final feature tensor (combine numeric + encoded categorical)
x = torch.tensor(node_features, dtype=torch.float)

# Construct PyG data object
graph_A = Data(x=x, edge_index=edge_index)

# print(f'data_GraphA : {graph_A}')
# print(f"Nodes: {graph_A.num_nodes}, Edges: {graph_A.num_edges}, Features: {graph_A.num_features}")


# Combine all groups back into one DataFrame
# df_grouping = concat(grouped_list, ignore_index=True)
# df_grouping.to_csv("/data/geolocation-data/processed/result-site-GROUPING.csv", index=False)